# ML Bottleneck Classifier

Trains a RandomForestClassifier on labelled eBPF performance data.

Each row = one (pid, cpu) observation  
Target = label (cpu_bound, memory_bound, normal_idle, normal_light, normal_mixed, …)

Notebook flow:
1. Imports & config
2. Load CSV & derive features
3. Select Tier 1 features & build model matrix
4. Train model
5. Evaluate

In [23]:
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

CSV_PATH = 'src/perf_metrics.csv'
TEST_SIZE = 0.20
RANDOM_STATE = 42

TIER1_FEATURES = [
    'involuntary_pct',
    'total_runtime_ns',
    'avg_runq_latency_ns',
    'total_faults',
    'total_alloc_bytes',
    'large_page_allocs',
    'avg_syscall_latency_ns',
    'read_bytes',
    'write_bytes',
    'voluntary_switches',
    'mutex_contentions',
    'avg_mutex_wait_ns',

    # ADDING THESE ↓
    'ctx_switches',
    'involuntary_switches',
    'mem_intensity'
]

print("Config loaded")

Config loaded


In [24]:
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} rows")

# ── Feature Engineering ─────────────────────────────────────────────

# involuntary_pct → CPU preemption rate
if 'involuntary_pct' not in df.columns:
    if 'involuntary_switches' in df.columns and 'ctx_switches' in df.columns:
        df['involuntary_pct'] = np.where(
            df['ctx_switches'] > 0,
            df['involuntary_switches'] / df['ctx_switches'] * 100,
            0.0
        )
    else:
        df['involuntary_pct'] = 0.0

# total_faults → memory pressure
if 'total_faults' not in df.columns:
    minor = df['minor_faults'] if 'minor_faults' in df.columns else pd.Series(0, index=df.index)
    kernel = df['kernel_faults'] if 'kernel_faults' in df.columns else pd.Series(0, index=df.index)
    df['total_faults'] = minor + kernel

# avg_mutex_wait_ns → normalize lock wait time
if 'avg_mutex_wait_ns' not in df.columns:
    if 'avg_mutex_wait_us' in df.columns:
        df['avg_mutex_wait_ns'] = df['avg_mutex_wait_us'] * 1000
    else:
        df['avg_mutex_wait_ns'] = 0.0

# NEW: mem_intensity → memory usage per unit time
df['mem_intensity'] = df['total_alloc_bytes'] / (df['total_runtime_ns'] + 1)

# ── Label Cleaning ─────────────────────────────────────────────────

df['label'] = df['label'].fillna('unknown').astype(str).str.strip()

# Merge similar normal classes
df['label'] = df['label'].replace({
    'normal_idle': 'normal',
    'normal_light': 'normal',
    'normal_mixed': 'normal'
})

print("\nLabel distribution:")
print(df['label'].value_counts())

Loaded 53758 rows

Label distribution:
label
normal          21043
memory_bound    17679
cpu_bound       15036
Name: count, dtype: int64


In [25]:
missing = [f for f in TIER1_FEATURES if f not in df.columns]
assert len(missing) == 0, f"Missing columns: {missing}"

df_model = df[TIER1_FEATURES + ['label']].copy()

df_model = df_model[df_model['label'] != 'unknown']
df_model = df_model.dropna()

df_model[TIER1_FEATURES] = df_model[TIER1_FEATURES].fillna(0)

X = df_model[TIER1_FEATURES].values
y = df_model['label'].values

print("Dataset ready:", X.shape)

Dataset ready: (53758, 15)


In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

clf = RandomForestClassifier(
    n_estimators=150,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight='balanced'
)

clf.fit(X_train, y_train)

print("Model trained")

Model trained


In [27]:
y_pred = clf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
labels = sorted(set(y))
print(labels)
print(confusion_matrix(y_test, y_pred, labels=labels))

print("\nFeature Importance:")
importances = pd.Series(clf.feature_importances_, index=TIER1_FEATURES)
print(importances.sort_values(ascending=False))

Accuracy: 0.6886

Classification Report:
              precision    recall  f1-score   support

   cpu_bound       0.89      0.86      0.88      3007
memory_bound       0.59      0.57      0.58      3536
      normal       0.64      0.66      0.65      4209

    accuracy                           0.69     10752
   macro avg       0.70      0.70      0.70     10752
weighted avg       0.69      0.69      0.69     10752


Confusion Matrix:
['cpu_bound', 'memory_bound', 'normal']
[[2593  187  227]
 [ 145 2033 1358]
 [ 178 1253 2778]]

Feature Importance:
avg_runq_latency_ns       0.416255
total_runtime_ns          0.220539
avg_syscall_latency_ns    0.060307
ctx_switches              0.053193
involuntary_pct           0.050839
voluntary_switches        0.049611
involuntary_switches      0.042957
total_faults              0.023874
total_alloc_bytes         0.020177
mem_intensity             0.019708
write_bytes               0.015156
read_bytes                0.015128
avg_mutex_wait_ns      